# مسئلهٔ ۱ — V2-MP / manifest چندموقعیتی برای آموزش

تحلیل full-MP4 نشان داد مدل A2 در inference با رخدادهایی روبه‌رو می‌شود که در موقعیت‌های متفاوتِ پنجره قرار دارند، در حالی‌که positive window آموزش قبلی تقریباً همیشه رخداد را در یک موقعیت ثابت داشت.

در این نسخه فقط sequenceهای train چندبرابر می‌شوند. برای هر مثبت، سه پنجره ساخته می‌شود که رخداد هدف‌گذاری‌شده در ثانیهٔ ۱٫۰، ۲٫۵ و ۴٫۰ از پنجره باشد. برای هر منفی نیز سه پنجرهٔ هم‌زمانی نسبی ساخته می‌شود. validation همان ۱۲۰ sequence ثابت قبلی باقی می‌ماند.

In [1]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd

DATA_ROOT = Path(r'P:\NexarCollisionData')
VIDEO_MANIFEST_PATH = DATA_ROOT / 'video_manifest_v2.csv'
BASE_SEQUENCE_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2.csv'
MULTIPOS_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2_multipos.csv'
EXCLUDED_PATH = DATA_ROOT / 'sequence_manifest_v2_multipos_excluded.csv'
SUMMARY_PATH = DATA_ROOT / 'sequence_manifest_v2_multipos_summary.json'

WINDOW_SECONDS = 5.0
NUM_FRAMES = 16
TRAIN_EVENT_POSITIONS_SECONDS = (1.0, 2.5, 4.0)
MATCHING_SEED = 420
VERSION = 'V2-MP-3positions'

assert VIDEO_MANIFEST_PATH.exists(), 'Run notebook 07 first.'
assert BASE_SEQUENCE_MANIFEST_PATH.exists(), 'Run notebook 08 first.'

In [2]:
manifest = pd.read_csv(VIDEO_MANIFEST_PATH).copy()
base_sequences = pd.read_csv(BASE_SEQUENCE_MANIFEST_PATH).copy()
manifest['video_id'] = manifest['video_id'].astype(str)
manifest['label'] = manifest['label'].astype(int)
manifest['duration'] = pd.to_numeric(manifest['duration'], errors='coerce')
manifest['time_of_event'] = pd.to_numeric(manifest['time_of_event'], errors='coerce')
manifest['is_valid'] = manifest['is_valid'].astype(str).str.lower().eq('true')
base_sequences['video_id'] = base_sequences['video_id'].astype(str)

eligible = manifest.loc[manifest['is_valid'] & manifest['duration'].ge(WINDOW_SECONDS)].copy()
excluded = manifest.loc[~manifest.index.isin(eligible.index)].copy()
excluded['sequence_exclusion_reason'] = np.where(~excluded['is_valid'], excluded['error_reason'].fillna('invalid_video'), 'video_shorter_than_5_seconds')
excluded.to_csv(EXCLUDED_PATH, index=False)

assert len(eligible) == 600
assert eligible.groupby(['split', 'label']).size().to_dict() == {('train', 0): 240, ('train', 1): 240, ('validation', 0): 60, ('validation', 1): 60}
assert (eligible['label'].eq(1) == eligible['time_of_event'].notna()).all()
display(pd.crosstab(eligible['split'], eligible['label']))

label,0,1
split,,
train,240,240
validation,60,60


In [3]:
def fixed_length_window(requested_start: float, duration: float) -> tuple[float, float, str]:
    if not np.isfinite(duration) or duration < WINDOW_SECONDS:
        raise ValueError('video_shorter_than_required_window')
    max_start = duration - WINDOW_SECONDS
    start = float(np.clip(requested_start, 0.0, max_start))
    if np.isclose(start, requested_start):
        policy = 'requested_window_kept'
    elif requested_start < 0:
        policy = 'shifted_to_video_start'
    else:
        policy = 'shifted_to_video_end'
    return start, start + WINDOW_SECONDS, policy

def add_timestamps(record: dict) -> dict:
    timestamps = np.linspace(record['window_start'], record['window_end'], num=NUM_FRAMES, endpoint=False)
    for index, timestamp in enumerate(timestamps):
        record[f'timestamp_{index:02d}'] = float(timestamp)
    return record

def positive_record(row: pd.Series, target_event_position: float, variant_index: int) -> dict:
    event_time = float(row.time_of_event)
    duration = float(row.duration)
    start, end, boundary_policy = fixed_length_window(event_time - target_event_position, duration)
    if not (start <= event_time <= end):
        raise ValueError('positive_event_outside_generated_window')
    actual_event_position = event_time - start
    record = {
        'sequence_id': f'{VERSION}_{row.video_id}_p{variant_index}',
        'base_video_id': row.video_id,
        'video_id': row.video_id,
        'video_path': row.video_path,
        'label': 1,
        'split': 'train',
        'sequence_variant': f'event_position_{target_event_position:.1f}s',
        'time_of_event': event_time,
        'duration': duration,
        'window_start': start,
        'window_end': end,
        'window_length_seconds': WINDOW_SECONDS,
        'window_center_ratio': (start + WINDOW_SECONDS / 2) / duration,
        'target_event_position_seconds': target_event_position,
        'actual_event_position_seconds': actual_event_position,
        'window_policy': f'positive_multipos;{boundary_policy}',
        'matched_positive_video_id': row.video_id,
        'matching_seed': MATCHING_SEED,
        'sampling_seed': MATCHING_SEED,
        'num_frames': NUM_FRAMES,
        'preprocessing_version': f'{VERSION}_manifest',
        'weather': row.weather,
        'light_conditions': row.light_conditions,
        'scene': row.scene,
    }
    return add_timestamps(record)

In [4]:
train_positive_rows = eligible.loc[eligible['split'].eq('train') & eligible['label'].eq(1)].copy()
train_negative_rows = eligible.loc[eligible['split'].eq('train') & eligible['label'].eq(0)].copy()
positive_records = []
for variant_index, target_position in enumerate(TRAIN_EVENT_POSITIONS_SECONDS):
    positive_records.extend(positive_record(row, target_position, variant_index) for _, row in train_positive_rows.iterrows())
positive_sequences = pd.DataFrame(positive_records)
assert len(positive_sequences) == 240 * len(TRAIN_EVENT_POSITIONS_SECONDS)
assert ((positive_sequences['window_start'] <= positive_sequences['time_of_event']) & (positive_sequences['time_of_event'] <= positive_sequences['window_end'])).all()

def build_negative_records(variant_index: int) -> list[dict]:
    positives = positive_sequences.loc[positive_sequences['sequence_id'].str.endswith(f'_p{variant_index}')].sort_values('video_id', key=lambda values: values.astype(int)).reset_index(drop=True)
    negatives = train_negative_rows.sort_values('video_id', key=lambda values: values.astype(int)).reset_index(drop=True)
    assert len(positives) == len(negatives) == 240
    rng = np.random.default_rng(MATCHING_SEED + variant_index)
    matched_indices = rng.permutation(len(positives))
    records = []
    for negative_index, (_, negative) in enumerate(negatives.iterrows()):
        matched = positives.iloc[matched_indices[negative_index]]
        duration = float(negative.duration)
        requested_start = float(matched.window_center_ratio) * duration - WINDOW_SECONDS / 2
        start, end, boundary_policy = fixed_length_window(requested_start, duration)
        record = {
            'sequence_id': f'{VERSION}_{negative.video_id}_n{variant_index}',
            'base_video_id': negative.video_id,
            'video_id': negative.video_id,
            'video_path': negative.video_path,
            'label': 0,
            'split': 'train',
            'sequence_variant': f'negative_match_position_{variant_index}',
            'time_of_event': np.nan,
            'duration': duration,
            'window_start': start,
            'window_end': end,
            'window_length_seconds': WINDOW_SECONDS,
            'window_center_ratio': (start + WINDOW_SECONDS / 2) / duration,
            'target_event_position_seconds': float(matched.target_event_position_seconds),
            'actual_event_position_seconds': np.nan,
            'window_policy': f'negative_matched_relative_position;{boundary_policy}',
            'matched_positive_video_id': matched.video_id,
            'matching_seed': MATCHING_SEED + variant_index,
            'sampling_seed': MATCHING_SEED,
            'num_frames': NUM_FRAMES,
            'preprocessing_version': f'{VERSION}_manifest',
            'weather': negative.weather,
            'light_conditions': negative.light_conditions,
            'scene': negative.scene,
        }
        records.append(add_timestamps(record))
    return records

negative_records = []
for variant_index in range(len(TRAIN_EVENT_POSITIONS_SECONDS)):
    negative_records.extend(build_negative_records(variant_index))
negative_sequences = pd.DataFrame(negative_records)
assert len(negative_sequences) == 240 * len(TRAIN_EVENT_POSITIONS_SECONDS)
display(pd.crosstab(positive_sequences['sequence_variant'], positive_sequences['label']))

label,1
sequence_variant,
event_position_1.0s,240
event_position_2.5s,240
event_position_4.0s,240


In [5]:
timestamp_columns = [f'timestamp_{index:02d}' for index in range(NUM_FRAMES)]
validation_base = base_sequences.loc[base_sequences['split'].eq('validation')].copy()
validation_base['base_video_id'] = validation_base['video_id'].astype(str)
validation_base['video_id'] = validation_base['video_id'].astype(str)
validation_base['sequence_id'] = [f'{VERSION}_{video_id}_validation' for video_id in validation_base['video_id']]
validation_base['sequence_variant'] = 'validation_fixed_v2_w2'
validation_base['target_event_position_seconds'] = np.where(validation_base['label'].eq(1), 3.0, np.nan)
validation_base['actual_event_position_seconds'] = np.where(validation_base['label'].eq(1), validation_base['time_of_event'] - validation_base['window_start'], np.nan)
validation_base['preprocessing_version'] = f'{VERSION}_manifest_validation'

sequence_manifest = pd.concat([positive_sequences, negative_sequences, validation_base], ignore_index=True, sort=False)
sequence_manifest = sequence_manifest.sort_values(['split', 'video_id', 'sequence_variant'], key=lambda values: values.astype(int) if values.name == 'video_id' else values).reset_index(drop=True)

assert len(sequence_manifest) == 1560
assert sequence_manifest['sequence_id'].is_unique
assert sequence_manifest.loc[sequence_manifest['split'].eq('train')].groupby('label').size().to_dict() == {0: 720, 1: 720}
assert sequence_manifest.loc[sequence_manifest['split'].eq('validation')].groupby('label').size().to_dict() == {0: 60, 1: 60}
assert np.allclose(sequence_manifest['window_length_seconds'], WINDOW_SECONDS)
assert (sequence_manifest['window_start'] >= -1e-8).all()
assert (sequence_manifest['window_end'] <= sequence_manifest['duration'] + 1e-8).all()
assert (sequence_manifest[timestamp_columns].diff(axis=1).iloc[:, 1:] > 0).all().all()
positive_check = sequence_manifest.loc[sequence_manifest['label'].eq(1)]
assert ((positive_check['window_start'] <= positive_check['time_of_event']) & (positive_check['time_of_event'] <= positive_check['window_end'])).all()

sequence_manifest.to_csv(MULTIPOS_MANIFEST_PATH, index=False)
summary = {
    'version': VERSION,
    'train_event_positions_seconds': list(TRAIN_EVENT_POSITIONS_SECONDS),
    'train_sequences': int(sequence_manifest['split'].eq('train').sum()),
    'validation_sequences': int(sequence_manifest['split'].eq('validation').sum()),
    'train_class_counts': {str(key): int(value) for key, value in sequence_manifest.loc[sequence_manifest['split'].eq('train'), 'label'].value_counts().sort_index().items()},
    'validation_class_counts': {str(key): int(value) for key, value in sequence_manifest.loc[sequence_manifest['split'].eq('validation'), 'label'].value_counts().sort_index().items()},
    'excluded_videos': int(len(excluded)),
    'timestamp_unit': 'seconds from video start',
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Multi-position sequence manifest: {MULTIPOS_MANIFEST_PATH}')
print(f'Summary: {SUMMARY_PATH}')
display(pd.crosstab(sequence_manifest['split'], sequence_manifest['label']))
display(sequence_manifest.loc[sequence_manifest['label'].eq(1), ['sequence_id', 'video_id', 'sequence_variant', 'time_of_event', 'window_start', 'window_end', 'actual_event_position_seconds']].head(12))

Multi-position sequence manifest: P:\NexarCollisionData\sequence_manifest_v2_multipos.csv
Summary: P:\NexarCollisionData\sequence_manifest_v2_multipos_summary.json


label,0,1
split,,
train,720,720
validation,60,60


,sequence_id,video_id,sequence_variant,time_of_event,window_start,window_end,actual_event_position_seconds
0,V2-MP-3positions_0_p0,0,event_position_1.0s,20.760,19.760,24.760,1.0
1,V2-MP-3positions_0_p1,0,event_position_2.5s,20.760,18.260,23.260,2.5
2,V2-MP-3positions_0_p2,0,event_position_4.0s,20.760,16.760,21.760,4.0
3,V2-MP-3positions_4_p0,4,event_position_1.0s,19.367,18.367,23.367,1.0
4,V2-MP-3positions_4_p1,4,event_position_2.5s,19.367,16.867,21.867,2.5
5,V2-MP-3positions_4_p2,4,event_position_4.0s,19.367,15.367,20.367,4.0
6,V2-MP-3positions_5_p0,5,event_position_1.0s,20.874,19.874,24.874,1.0
7,V2-MP-3positions_5_p1,5,event_position_2.5s,20.874,18.374,23.374,2.5
8,V2-MP-3positions_5_p2,5,event_position_4.0s,20.874,16.874,21.874,4.0
9,V2-MP-3positions_6_p0,6,event_position_1.0s,19.233,18.233,23.233,1.0


## شرط پایان این مرحله

باید ۱۴۴۰ sequence متوازن train و ۱۲۰ sequence validation ثابت داشته باشیم. سپس فقط frameهای جدید train cache می‌شوند؛ cache validation قبلی قابل reuse است. مدل A2-MP روی همین manifest آموزش می‌بیند و دوباره با sliding-window full-MP4 سنجیده می‌شود.